---
## Stage 14: Identity Resolution Inference & Merge Decision

**วัตถุประสงค์:** รัน inference บน candidate pairs → ตัดสินใจ merge/review/reject

**Input:** `candidate_pairs`, `model.pt`, `scaler.pkl`, `calibrator.pkl`  
**Output:** `predictions.parquet` พร้อม 3-level decisions

| Sub-step | หน้าที่ |
|----------|--------|
| 14.1 | Load Artifacts |
| 14.2 | Compute Features + Predict |
| 14.3 | Apply 3-Level Threshold |
| 14.4 | Save Predictions |

### Step 14.1: Load Artifacts

In [ ]:
# --- 14.1 Load Artifacts ---
# ถ้ารันต่อจาก cell ที่แล้ว → model, scaler, calibrator อยู่ใน memory แล้ว
# ถ้ารันใหม่ → โหลดจากไฟล์

print("📊 Step 14.1: Load Artifacts")
print("=" * 60)

try:
    _ = model, scaler, calibrator, feature_cols
    print("  ✅ Artifacts already in memory")
except NameError:
    # โหลดจากไฟล์
    model = IdentityMLP(input_dim=len(feature_cols)).to(device)
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'model.pt'), weights_only=True))
    model.eval()
    with open(os.path.join(OUTPUT_DIR, 'scaler.pkl'), 'rb') as f:
        scaler = pickle.load(f)
    with open(os.path.join(OUTPUT_DIR, 'calibrator.pkl'), 'rb') as f:
        calibrator = pickle.load(f)
    with open(os.path.join(OUTPUT_DIR, 'feature_cols.pkl'), 'rb') as f:
        feature_cols = pickle.load(f)
    print("  ✅ Artifacts loaded from files")

print(f"  Model       : IdentityMLP ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Features    : {len(feature_cols)} columns")
print(f"\n✅ Step 14.1 เสร็จ")

### Step 14.2: Compute Features for Candidates + Predict

In [ ]:
# --- 14.2 Features + Predict ---
print("📊 Step 14.2: Inference on Candidate Pairs")
print("=" * 60)
print(f"  Candidate pairs: {len(candidate_pairs):,}")

# คำนวณ features สำหรับ candidate pairs (ใช้ logic เดียวกับ Stage 9)
infer_rows = []
for _, pair in candidate_pairs.iterrows():
    row = {}
    id_a, id_b = pair['profile_id_a'], pair['profile_id_b']
    
    if id_a in profile_lookup.index and id_b in profile_lookup.index:
        r_a = profile_lookup.loc[id_a]
        r_b = profile_lookup.loc[id_b]
        if isinstance(r_a, pd.DataFrame): r_a = r_a.iloc[0]
        if isinstance(r_b, pd.DataFrame): r_b = r_b.iloc[0]
        
        for col, prefix in [('userName_clean','username'),('fullName_clean','fullname'),('bio_clean','bio')]:
            va, vb = str(r_a.get(col,'')), str(r_b.get(col,''))
            for method in ['jaro','token_sort','levenshtein']:
                row[f'{prefix}_{method}'] = string_sim(va, vb, method)
            row[f'{prefix}_both_empty'] = 1.0 if (len(va)==0 and len(vb)==0) else 0.0
        
        # TF-IDF, URL, meta features
        idx_a = key_to_idx.get(id_a); idx_b = key_to_idx.get(id_b)
        if idx_a is not None and idx_b is not None:
            row['bio_tfidf_cosine'] = float(sk_cosine(tfidf_matrix[idx_a:idx_a+1], tfidf_matrix[idx_b:idx_b+1])[0][0])
        else:
            row['bio_tfidf_cosine'] = 0.0
        
        ua, ub = str(r_a.get('externalUrl_clean','')), str(r_b.get('externalUrl_clean',''))
        da, db = str(r_a.get('url_domain','')), str(r_b.get('url_domain',''))
        row['url_exact_match'] = 1.0 if (ua and ub and ua==ub and len(ua)>0) else 0.0
        row['url_domain_match'] = 1.0 if (da and db and da==db and len(da)>0) else 0.0
        row['same_platform'] = 1.0 if r_a.get('platform','')==r_b.get('platform','') else 0.0
        la, lb = str(r_a.get('location_clean','')), str(r_b.get('location_clean',''))
        row['location_sim'] = string_sim(la, lb, 'jaro') if (la and lb) else 0.0
    else:
        for col in feature_cols:
            row[col] = 0.0
    infer_rows.append(row)

infer_features = pd.DataFrame(infer_rows)
# ตรวจ feature columns ครบ
for col in feature_cols:
    if col not in infer_features.columns:
        infer_features[col] = 0.0

X_infer = scaler.transform(infer_features[feature_cols].values)
X_tensor = torch.FloatTensor(X_infer).to(device)

model.eval()
with torch.no_grad():
    raw_probs = torch.sigmoid(model(X_tensor)).cpu().numpy()
cal_probs_infer = calibrator.predict(raw_probs)

print(f"  Predictions computed: {len(cal_probs_infer):,}")
print(f"  Prob distribution: mean={cal_probs_infer.mean():.4f}, std={cal_probs_infer.std():.4f}")
print(f"\n✅ Step 14.2 เสร็จ")

### Step 14.3-14.4: Apply 3-Level Threshold & Save

In [ ]:
# --- 14.3-14.4 Apply Threshold & Save ---
predictions_df = candidate_pairs.copy()
predictions_df['probability'] = cal_probs_infer
predictions_df['decision'] = predictions_df['probability'].apply(get_decision)

auto_merge = predictions_df[predictions_df['decision'] == 'MATCH']
review_queue = predictions_df[predictions_df['decision'] == 'POSSIBLE_MATCH']
no_match = predictions_df[predictions_df['decision'] == 'NO_MATCH']

print("=" * 60)
print("📊 STAGE 14 SUMMARY — Inference & Merge Decision")
print("=" * 60)
print(f"  Total predictions  : {len(predictions_df):,}")
print(f"  MATCH (≥90%)       : {len(auto_merge):,} → auto-merge")
print(f"  POSSIBLE (70-89%)  : {len(review_queue):,} → human review")
print(f"  NO_MATCH (<70%)    : {len(no_match):,} → keep separate")

if len(auto_merge) > 0:
    print(f"\n  🔍 Top 5 MATCH pairs:")
    for _, r in auto_merge.nlargest(5, 'probability').iterrows():
        print(f"    prob={r['probability']:.3f} | {r['profile_id_a'][:30]} ↔ {r['profile_id_b'][:30]}")

# Save
predictions_df.to_csv(os.path.join(OUTPUT_DIR, 'predictions.csv'), index=False)
print(f"\n  💾 Saved: predictions.csv")
print(f"\n{'='*60}")
print(f"✅ Stage 14 COMPLETE")
print(f"{'='*60}")